In [1]:
import pandas as pd
import optuna
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import SMOTE

In [2]:
# Load the dataset
df = pd.read_csv(r'C:\Users\khiew\Downloads\FYP Reduced (Secondly) Dataset.csv')

In [4]:
# Drop diseases with less than 50 instances
disease_counts = df['diseases'].value_counts()
valid_diseases = disease_counts[disease_counts >= 750].index
df = df[df['diseases'].isin(valid_diseases)]

# Assuming that the target variable is 'diseases' and all other variables are input features
X = df.drop('diseases', axis=1)
y = df['diseases']

# Encode the target variable (diseases) if it's a categorical variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
print("Number of remaining classes in training set:", len(np.unique(y_train)))
print("Number of rows left:", len(df))

Number of remaining classes in training set: 114
Number of rows left: 114312


In [5]:
# Apply SMOTE for class balancing in the training set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Print the new class distribution after SMOTE
print("Class distribution after SMOTE:")
print(pd.Series(y_train_resampled).value_counts())
print("Number of remaining classes in training set:", len(np.unique(y_train_resampled)))
# Print the number of rows in the resampled training set
print("Number of rows in the resampled training set:", len(X_train_resampled))

Class distribution after SMOTE:
20     1002
9      1002
66     1002
80     1002
1      1002
       ... 
40     1002
106    1002
54     1002
102    1002
88     1002
Name: count, Length: 114, dtype: int64
Number of remaining classes in training set: 114
Number of rows in the resampled training set: 114228


In [6]:
# Optuna optimization function
def objective(trial):
    # Define the hyperparameters to tune
    n_estimators = trial.suggest_int('n_estimators', 50, 150)
    max_depth = trial.suggest_int('max_depth', 10, 50)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])

    # Create RandomForestClassifier with hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42
    )

    # Perform cross-validation (using 5-fold by default)
    score = cross_val_score(model, X_train_resampled, y_train_resampled, cv=5, scoring='accuracy')
    accuracy = score.mean()

    # Log the hyperparameters and accuracy for this trial
    print(f"Trial {trial.number}: n_estimators={n_estimators}, max_depth={max_depth}, "
          f"min_samples_split={min_samples_split}, min_samples_leaf={min_samples_leaf}, "
          f"max_features={max_features}, Accuracy={accuracy:.4f}")

    return accuracy

In [8]:
# Create Optuna study for optimization with persistent storage
study = optuna.create_study(
    direction='maximize',  # Assuming you are maximizing accuracy
    study_name="randomforest_diseases_symptoms_dropextremelymore750withSMOTE_SecondReduction_study", 
    storage=r"sqlite:///C:/Users/khiew/Downloads/randomforest.db", 
    load_if_exists=True  # Load the study if it already exists, to resume from the last trial
)

# Optimize the study with your objective function, you can adjust the n_trials as needed
study.optimize(objective, n_trials=20)
# Print the best trial and hyperparameters
print("\nBest Trial:")
print(study.best_trial)
print("Best Hyperparameters:")
print(study.best_trial.params)

[I 2025-04-22 19:10:08,667] A new study created in RDB with name: randomforest_diseases_symptoms_dropextremelymore750withSMOTE_SecondReduction_study
[I 2025-04-22 19:10:34,330] Trial 0 finished with value: 0.2731992740529733 and parameters: {'n_estimators': 105, 'max_depth': 48, 'min_samples_split': 20, 'min_samples_leaf': 17, 'max_features': None}. Best is trial 0 with value: 0.2731992740529733.


Trial 0: n_estimators=105, max_depth=48, min_samples_split=20, min_samples_leaf=17, max_features=None, Accuracy=0.2732


[I 2025-04-22 19:11:11,587] Trial 1 finished with value: 0.27778658313918847 and parameters: {'n_estimators': 149, 'max_depth': 32, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': None}. Best is trial 1 with value: 0.27778658313918847.


Trial 1: n_estimators=149, max_depth=32, min_samples_split=4, min_samples_leaf=3, max_features=None, Accuracy=0.2778


[I 2025-04-22 19:11:21,199] Trial 2 finished with value: 0.2781017245140974 and parameters: {'n_estimators': 68, 'max_depth': 48, 'min_samples_split': 4, 'min_samples_leaf': 7, 'max_features': 'sqrt'}. Best is trial 2 with value: 0.2781017245140974.


...

Trial 17: n_estimators=85, max_depth=36, min_samples_split=8, min_samples_leaf=8, max_features=sqrt, Accuracy=0.2783


[I 2025-04-22 19:15:12,727] Trial 18 finished with value: 0.2781892675743553 and parameters: {'n_estimators': 89, 'max_depth': 45, 'min_samples_split': 12, 'min_samples_leaf': 14, 'max_features': 'sqrt'}. Best is trial 16 with value: 0.27878457502245524.


Trial 18: n_estimators=89, max_depth=45, min_samples_split=12, min_samples_leaf=14, max_features=sqrt, Accuracy=0.2782


[I 2025-04-22 19:15:24,876] Trial 19 finished with value: 0.27863575017224484 and parameters: {'n_estimators': 83, 'max_depth': 37, 'min_samples_split': 7, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 16 with value: 0.27878457502245524.


Trial 19: n_estimators=83, max_depth=37, min_samples_split=7, min_samples_leaf=5, max_features=sqrt, Accuracy=0.2786

Best Trial:
FrozenTrial(number=16, state=TrialState.COMPLETE, values=[0.27878457502245524], datetime_start=datetime.datetime(2025, 4, 22, 19, 14, 36, 283838), datetime_complete=datetime.datetime(2025, 4, 22, 19, 14, 48, 154314), params={'n_estimators': 85, 'max_depth': 37, 'min_samples_split': 12, 'min_samples_leaf': 9, 'max_features': 'sqrt'}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=150, log=False, low=50, step=1), 'max_depth': IntDistribution(high=50, log=False, low=10, step=1), 'min_samples_split': IntDistribution(high=20, log=False, low=2, step=1), 'min_samples_leaf': IntDistribution(high=20, log=False, low=1, step=1), 'max_features': CategoricalDistribution(choices=('sqrt', 'log2', None))}, trial_id=434, value=None)
Best Hyperparameters:
{'n_estimators': 85, 'max_depth': 37, 'min_samples_split': 12